[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/cours/seance3_cours.ipynb)

# Séance 4.3 — Arbres et forêts — ce qui fait vraiment la prédiction

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- lire un arbre de décision comme une suite de règles métier
- choisir la complexité d'un modèle par validation croisée, sans toucher au test
- dire pourquoi l'importance native d'un modèle est biaisée
- mesurer l'importance d'une variable par permutation, sur le jeu de test
- montrer le **sens** d'un effet avec une dépendance partielle

## La question que le comité pose toujours

En séance 4.2, le modèle a désigné 1 073 abonnés à rappeler. En comité, la
question suivante tombe immanquablement :

> *« D'accord. Mais **qu'est-ce qui** fait qu'un client part ? »*

Un modèle qui prédit sans expliquer ne se déploie pas : personne n'engage un
budget sur une boîte noire. Cette séance répond à la question — et montre au
passage que **la première réponse qu'on obtient est fausse**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, roc_auc_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")
tel = tel.dropna(subset=["total"])

y = tel["churn"]
# .astype(float) : les dependances partielles refusent les colonnes
# entieres, et les 0/1 de get_dummies sont des booleens
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True).astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(X.shape[1], "variables")

## 1. Un arbre se lit

Un arbre de décision pose des questions en cascade. Sa force n'est pas sa
performance : c'est qu'on peut le **lire**.

In [ ]:
arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)

print(export_text(arbre, feature_names=list(X.columns)))

Lisez la première ligne à voix haute : *« le client est-il au contrat
mensuel ? »* L'arbre a choisi tout seul, parmi quatorze variables, de couper
d'abord sur celle-là — exactement ce que le tableau de la séance 4.2 montrait.

C'est une suite de **règles métier**, transposable telle quelle dans une note
de service.

In [ ]:
plt.figure(figsize=(7, 4))
plot_tree(arbre, feature_names=list(X.columns), class_names=["reste", "part"],
          max_depth=2, filled=True, fontsize=7)
plt.show()

## 2. Une forêt fait-elle mieux ?

Une **forêt aléatoire** entraîne deux cents arbres sur des échantillons un peu
différents, puis les fait voter. Plus de modèle, plus de calcul.

In [ ]:
foret = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)

for nom, m in {"arbre (prof. 3)": arbre, "foret (200 arbres)": foret}.items():
    p = m.predict_proba(X_te)[:, 1]
    print(f"{nom:<20} justesse {accuracy_score(y_te, m.predict(X_te)):.3f}  "
          f"AUC {roc_auc_score(y_te, p):.3f}")

**AUC 0,813 pour l'arbre, 0,805 pour la forêt.** Le petit arbre gagne.

N'en concluez pas que les forêts ne servent à rien : sur d'autres jeux, elles
gagnent nettement. La leçon est ailleurs — **on mesure, on ne suppose pas**.
Et à performance égale, le modèle qui se lit l'emporte toujours, parce qu'il
peut être discuté, corrigé et appliqué par des humains.

## 3. Choisir un réglage sans tricher

Quelle profondeur retenir ? Tentation : les essayer toutes sur le test et
garder la meilleure. **Impossible** — le test aurait servi à décider, il ne
mesurerait plus rien.

La **validation croisée** résout ça sans y toucher : on découpe le jeu
d'apprentissage en cinq, on entraîne sur quatre morceaux, on évalue sur le
cinquième, cinq fois.

In [ ]:
for profondeur in [2, 3, 4, 5, 8, None]:
    scores = cross_val_score(
        DecisionTreeClassifier(max_depth=profondeur, random_state=42),
        X_tr, y_tr, cv=5, scoring="roc_auc")
    print(f"profondeur {str(profondeur):<5} AUC {scores.mean():.3f}")

**Optimum à 4** (0,826), effondrement à 0,669 sans limite. La même courbe en
cloche qu'en séance 4.1 — trop simple ne capte rien, trop complexe apprend le
bruit.

Le test, lui, n'a pas été ouvert. Il reste disponible pour la mesure finale,
celle qu'on annonce.

## 4. Qu'est-ce qui fait la prédiction ?

### Première réponse : l'importance native

In [ ]:
native = pd.Series(foret.feature_importances_, index=X.columns)

native.sort_values(ascending=False).head(6).round(3)

La facture cumulée arrive en tête, suivie de la facture mensuelle. Conclusion
apparente : **c'est le montant qui fait partir les clients**.

Gardez cette phrase en tête trente secondes.

### Deuxième réponse : l'importance par permutation

Principe : on mélange une colonne au hasard — ce qui détruit son information —
et on regarde **combien le modèle perd**, sur le jeu de test. Une variable
utile fait chuter la performance ; une variable inutile ne change rien.

In [ ]:
perm = permutation_importance(foret, X_te, y_te, n_repeats=5,
                              random_state=42, scoring="roc_auc")

pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(6).round(4)

**Le classement est renversé.** `contrat_mensuel` passe premier ; `total`,
première tout à l'heure, **disparaît du classement**.

### Pourquoi ?

`feature_importances_` compte combien de fois une variable a servi à couper.
Une variable continue à des milliers de valeurs distinctes offre des milliers
de coupures possibles ; une colonne 0/1 en offre **une seule**. Le calcul
favorise mécaniquement les premières — indépendamment de leur utilité réelle.

La permutation mesure autre chose : ce que le modèle **perd** quand on retire
l'information, sur des données qu'il n'a jamais vues.

> ⚠️ **Croyez la permutation.** Et remarquez qu'elle confirme le tableau de la
> séance 4.2 : 42,7 % de départs au contrat mensuel contre 2,8 % à deux ans.
> Trois chemins différents, une seule réponse — c'est ce qui rend la
> conclusion solide.

## 5. Une importance ne donne pas un sens

« Le contrat compte » ne se décide pas. Il faut savoir **dans quel sens** et
**de combien**. C'est ce que montre une **dépendance partielle** : la
probabilité prédite quand on fait varier une seule variable.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(foret, X_te, ["anc"], ax=ax)
plt.title("Risque de depart selon l'anciennete")
plt.show()

La courbe chute fortement sur les premiers mois puis s'aplatit : **le risque
se joue la première année**, et un client qui a passé deux ans ne part
pratiquement plus.

Voilà une phrase de note de direction : « l'effort de rétention doit porter
sur les douze premiers mois ; au-delà, il est dépensé pour rien. »

## 6. L'erreur qui ne prévient pas

In [ ]:
sur_appr = permutation_importance(foret, X_tr, y_tr, n_repeats=3,
                                  random_state=42, scoring="roc_auc")

print("mesuree sur l'apprentissage :")
print(pd.Series(sur_appr.importances_mean, index=X.columns).nlargest(3).round(4))

Mesurée sur l'apprentissage, l'importance récompense ce que le modèle a
**mémorisé**, pas ce qui l'aide à généraliser. Le classement change, et aucun
message d'erreur ne le signale.

> ⚠️ **Une importance se mesure toujours sur des données que le modèle n'a
> jamais vues.** C'est la règle de la séance 4.1, appliquée à
> l'interprétation.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| un arbre lisible | `DecisionTreeClassifier(max_depth=3)` |
| lire ses règles | `print(export_text(a, feature_names=list(X.columns)))` |
| le dessiner | `plot_tree(a, feature_names=..., filled=True)` |
| une forêt | `RandomForestClassifier(n_estimators=200, random_state=42)` |
| choisir un réglage sans toucher au test | `cross_val_score(modele, X_tr, y_tr, cv=5, scoring="roc_auc")` |
| l'importance native (biaisée) | `m.feature_importances_` |
| l'importance honnête | `permutation_importance(m, X_te, y_te, n_repeats=5)` |
| le sens de l'effet | `PartialDependenceDisplay.from_estimator(m, X_te, ["anc"])` |

## Les trois mesures d'importance

| Mesure | Ce qu'elle vaut | Son défaut |
|---|---|---|
| `feature_importances_` | gratuite, immédiate | **favorise les variables à nombreuses valeurs** — ici `total` et `mensuel` |
| permutation | mesurée sur le test, sur la vraie métrique | plus lente ; trompeuse si deux variables se dupliquent |
| dépendance partielle | donne le **sens** et l'ampleur | une variable à la fois |

## Les trois phrases à retenir

1. **Un arbre de profondeur 3 fait ici mieux qu'une forêt de 200 arbres**
   (AUC 0,813 contre 0,805). Le modèle lisible n'est pas le modèle faible.

2. **L'importance native ment.** Elle classe `total` première ; la permutation
   la sort du classement et met le contrat mensuel en tête — ce que le tableau
   de la séance 4.2 disait déjà.

3. **Une importance ne donne pas un sens.** « Le contrat compte » ne se décide
   pas ; « passer un client au mensuel double son risque de départ » se décide.